# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # This is a mlcroissant.Metadata object
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.date_published}")
print(f"License: {metadata.license}")
print(f"Cite as: {metadata.cite_as}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We can list all RecordSets (tables) included in the dataset and detail their Fields and Columns (by `@id`).

In [ ]:
# List all RecordSets in the dataset, referencing them by @id
print("Available RecordSets (@id):\n----------------------")
record_sets = []
for recset in metadata.record_sets:
    record_sets.append(recset.id)
    print(f"@id: {recset.id}, name: {recset.name}, description: {recset.description}")
    # List all fields of this record set
    print("  Fields:")
    for field in recset.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type}, column: {field.column}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set by @id
# List of the record_set @ids obtained above - update with actual IDs from the printout
record_sets_ids = record_sets  # Already obtained in the previous cell
dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for: {record_set_id}, shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for: {record_set_id}")
        dataframes[record_set_id] = pd.DataFrame()


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Note:** Replace the `example_record_set_id`, `numeric_field_id`, and `group_field_id` with actual @ids from step 2 & 3 appropriate to your analysis.

In [ ]:
# Example: EDA for one record set
# Fill these in with valid values from your overview (step 2 output)
example_record_set_id = record_sets_ids[0] if record_sets_ids else None
if example_record_set_id and not dataframes[example_record_set_id].empty:
    df = dataframes[example_record_set_id]
    # Find numeric columns
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    print(f"Numeric columns available in DataFrame: {numeric_cols}")
    if numeric_cols:
        numeric_field = numeric_cols[0]  # or pick the field you're interested in
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a likely categorical field
        group_fields = [col for col in df.columns if "name" in col.lower() or "type" in col.lower() or "group" in col.lower()]
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
else:
    print("No non-empty record sets available for EDA. Check previous cells for data.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and not dataframes[example_record_set_id].empty:
    df = dataframes[example_record_set_id]
    if 'numeric_field' in locals() and numeric_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
        plt.title(f"Distribution of {numeric_field} in {example_record_set_id}")
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()
    # Scatter plot example (requires two numeric fields)
    if len(df.select_dtypes(include='number').columns) > 1:
        x, y = df.select_dtypes(include='number').columns[:2]
        plt.figure(figsize=(8,6))
        sns.scatterplot(data=df, x=x, y=y)
        plt.xlabel(x)
        plt.ylabel(y)
        plt.title(f"Scatter plot: {y} vs {x}")
        plt.show()
else:
    print("No valid data found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook showed how to load FAIR2-compliant datasets using the `mlcroissant` library, reference data entities by their `@id`, and perform basic analysis and visualization.
- You can further process each RecordSet and its fields based on domain knowledge of the dataset.